In [ ]:
# ============================================================
# 0. GPU Setting
# ============================================================

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [ ]:
# ============================================================
# 1. Libraries
# ============================================================

from pathlib import Path
import random
import time
import joblib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    import optuna
    print("Optuna already installed:", optuna.__version__)
except ImportError:
    !{sys.executable} -m pip install --user optuna
    import optuna
    print("Optuna installed:", optuna.__version__)

In [ ]:
# ============================================================
# 2. Project Paths
# ============================================================

PROJECT_DIR = Path("/home/u2022144048/projects/daechung_waterlevel_lstm")

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
METRIC_DIR = OUTPUT_DIR / "metrics"
FIGURE_DIR = OUTPUT_DIR / "figures"
LOG_DIR = PROJECT_DIR / "logs"

for d in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MODEL_DIR:", MODEL_DIR)

In [ ]:
# ============================================================
# 3. Reproducibility & Device Check
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("Device:", device)

if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 4. Load Data
# ============================================================

DATA_PATH = PROCESSED_DIR / "daechung_weather_level_hourly_2006_2025.csv"

df = pd.read_csv(DATA_PATH)

TIME_COL = "datetime"          # 네 시간 컬럼명이 tm이면 "tm"으로 바꾸기
TARGET_COL = "daechung_level"  # 대청댐 수위

df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values(TIME_COL).reset_index(drop=True)

print("Data shape:", df.shape)
print("Time range:", df[TIME_COL].min(), "~", df[TIME_COL].max())
print(df.head())

In [ ]:
# ============================================================
# 5. Feature Columns
# ============================================================

weather_stations = [
    "buyeo",
    "cheongju",
    "boeun",
    "daejeon",
    "geumsan",
    "chupungnyeong",
    "jeonju",
    "imsil"
]

weather_vars = ["rn", "hm", "ta", "pa"]

water_level_stations = [
    "sangyeeri",
    "iwondaegyo",
    "yulli",
    "hotanri",
    "jeokbyeokgyo",
    "sinyongdamgyo",
    "yeonhwagyo",
    "seongsanri"
]

meteo_cols = [
    f"{stn}_{var}"
    for stn in weather_stations
    for var in weather_vars
]

upstream_level_cols = [
    f"{stn}_level"
    for stn in water_level_stations
]

dam_cols = ["yongdam_level"]

# 과거 대청댐 수위 포함
feature_cols = meteo_cols + upstream_level_cols + dam_cols + [TARGET_COL]

print("Meteorological features:", len(meteo_cols))
print("Upstream level features:", len(upstream_level_cols))
print("Dam features:", len(dam_cols))
print("Past target included:", TARGET_COL)
print("Total input features:", len(feature_cols))
print(feature_cols)

In [ ]:
# ============================================================
# 6. Data Check
# ============================================================

required_cols = [TIME_COL] + feature_cols

missing_cols = [col for col in required_cols if col not in df.columns]

if len(missing_cols) > 0:
    print("Missing columns:")
    print(missing_cols)
    raise ValueError("Some required columns are missing.")
else:
    print("All required columns exist.")

missing = df[required_cols].isna().sum().sort_values(ascending=False)

print("\nColumns with missing values:")
print(missing[missing > 0])
print("\nTotal missing:", df[required_cols].isna().sum().sum())

In [ ]:
# ============================================================
# 7. Scaling
# ============================================================

TRAIN_END = "2021-01-01"

train_mask_raw = df[TIME_COL] < TRAIN_END
train_df_for_scaler = df.loc[train_mask_raw].copy()

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_X.fit(train_df_for_scaler[feature_cols].values)
scaler_y.fit(train_df_for_scaler[[TARGET_COL]].values)

X_all_scaled = scaler_X.transform(df[feature_cols].values)
y_all_scaled = scaler_y.transform(df[[TARGET_COL]].values)

print("X_all_scaled:", X_all_scaled.shape)
print("y_all_scaled:", y_all_scaled.shape)

print("Train scaled X min/max:",
      X_all_scaled[train_mask_raw.values].min(),
      X_all_scaled[train_mask_raw.values].max())

print("Train scaled y min/max:",
      y_all_scaled[train_mask_raw.values].min(),
      y_all_scaled[train_mask_raw.values].max())

In [ ]:
# ============================================================
# 8. Window Dataset Function
# ============================================================

def make_window_dataset(X_values, y_values, time_values, window_size=72, forecast_horizon=1):
    X_windows = []
    y_windows = []
    target_times = []

    n = len(X_values)

    for i in range(window_size, n - forecast_horizon + 1):
        X_windows.append(X_values[i - window_size:i, :])
        y_windows.append(y_values[i + forecast_horizon - 1, :])
        target_times.append(time_values[i + forecast_horizon - 1])

    X_windows = np.array(X_windows, dtype=np.float32)
    y_windows = np.array(y_windows, dtype=np.float32)
    target_times = np.array(target_times)

    return X_windows, y_windows, target_times

In [ ]:
# ============================================================
# 9. Create Window Dataset
# ============================================================

WINDOW_SIZE = 72
FORECAST_HORIZON = 1

X, y, target_times = make_window_dataset(
    X_values=X_all_scaled,
    y_values=y_all_scaled,
    time_values=df[TIME_COL].values,
    window_size=WINDOW_SIZE,
    forecast_horizon=FORECAST_HORIZON
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("target_times shape:", target_times.shape)

In [ ]:
# ============================================================
# 10. Train / Validation / Test Split
# ============================================================

target_times = pd.to_datetime(target_times)

train_mask = target_times < "2021-01-01"
val_mask = (target_times >= "2021-01-01") & (target_times < "2025-01-01")
test_mask = target_times >= "2025-01-01"

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

time_train = target_times[train_mask]
time_val = target_times[val_mask]
time_test = target_times[test_mask]

print("Train:", X_train.shape, y_train.shape, time_train.min(), "~", time_train.max())
print("Val:  ", X_val.shape, y_val.shape, time_val.min(), "~", time_val.max())
print("Test: ", X_test.shape, y_test.shape, time_test.min(), "~", time_test.max())

In [ ]:
# ============================================================
# 11. PyTorch Dataset
# ============================================================

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset = TimeSeriesDataset(X_val, y_val)
test_dataset = TimeSeriesDataset(X_test, y_test)

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
# ============================================================
# 12. LSTM Model
# ============================================================

class LSTMRegressor(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size=128,
        num_layers=2,
        dropout=0.2,
        dense_size=64
    ):
        super().__init__()

        self.num_layers = num_layers

        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout
        )

        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, dense_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(dense_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]

        x = self.dropout(last_hidden)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        output = self.fc2(x)

        return output


INPUT_SIZE = X_train.shape[2]
NUM_LAYERS = 2

print("Input size:", INPUT_SIZE)
print("Fixed LSTM layers:", NUM_LAYERS)

In [ ]:
# ============================================================
# 13. Metrics
# ============================================================

def nse_score(y_true, y_pred):
    y_true = np.array(y_true).reshape(-1)
    y_pred = np.array(y_pred).reshape(-1)

    denominator = np.sum((y_true - np.mean(y_true)) ** 2)

    if denominator == 0:
        return np.nan

    numerator = np.sum((y_true - y_pred) ** 2)
    return 1 - numerator / denominator


def kge_score(y_true, y_pred):
    y_true = np.array(y_true).reshape(-1)
    y_pred = np.array(y_pred).reshape(-1)

    if np.std(y_true) == 0 or np.mean(y_true) == 0:
        return np.nan

    r = np.corrcoef(y_true, y_pred)[0, 1]

    if np.isnan(r):
        return np.nan

    alpha = np.std(y_pred) / np.std(y_true)
    beta = np.mean(y_pred) / np.mean(y_true)

    return 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)


def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mae_score(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


def evaluate_predictions(y_true, y_pred, name=""):
    nse = nse_score(y_true, y_pred)
    kge = kge_score(y_true, y_pred)
    rmse = rmse_score(y_true, y_pred)
    mae = mae_score(y_true, y_pred)

    print(f"===== {name} Performance =====")
    print(f"NSE  : {nse:.4f}")
    print(f"KGE  : {kge:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")

    return {
        "NSE": nse,
        "KGE": kge,
        "RMSE": rmse,
        "MAE": mae
    }

In [ ]:
# ============================================================
# 14. Train & Validation Functions
# ============================================================

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    avg_loss = total_loss / len(train_loader.dataset)

    return avg_loss


def evaluate_loss(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            total_loss += loss.item() * X_batch.size(0)

    avg_loss = total_loss / len(data_loader.dataset)

    return avg_loss

In [ ]:
# ============================================================
# 15. Prediction Function
# ============================================================

def predict_model(model, data_loader, device):
    model.eval()

    preds = []
    trues = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)

            y_pred = model(X_batch)

            preds.append(y_pred.cpu().numpy())
            trues.append(y_batch.cpu().numpy())

    preds = np.vstack(preds)
    trues = np.vstack(trues)

    return trues, preds

In [ ]:
# ============================================================
# 16. Optuna Trial Training Function (5번만)
# ============================================================

def train_model_for_trial(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    max_epochs=80,
    patience=10
):
    best_val_loss = np.inf
    best_epoch = 0
    best_state_dict = None
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):

        train_loss = train_one_epoch(
            model=model,
            train_loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device
        )

        val_loss = evaluate_loss(
            model=model,
            data_loader=val_loader,
            criterion=criterion,
            device=device
        )

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0

            best_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return best_val_loss, best_epoch, best_state_dict

In [ ]:
# ============================================================
# 17. Optuna Objective with Best Trial Weight Saving
# ============================================================

BEST_TRIAL_MODEL_PATH = MODEL_DIR / "daechung_lstm_best_trial_window72_2layers_pastlevel.pt"

if BEST_TRIAL_MODEL_PATH.exists():
    BEST_TRIAL_MODEL_PATH.unlink()
    print("Old best trial checkpoint removed:", BEST_TRIAL_MODEL_PATH)


def objective(trial):

    start_time = time.time()

    num_layers = 2

    hidden_size = trial.suggest_categorical(
        "hidden_size",
        [64, 128, 256]
    )

    dropout = trial.suggest_float(
        "dropout",
        0.0,
        0.4
    )

    dense_size = trial.suggest_categorical(
        "dense_size",
        [32, 64, 128]
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-3,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [128, 256, 512]
    )

    patience = trial.suggest_categorical(
        "patience",
        [8, 10, 15]
    )

    train_loader_trial = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    val_loader_trial = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    model_trial = LSTMRegressor(
        input_size=INPUT_SIZE,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        dense_size=dense_size
    ).to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model_trial.parameters(),
        lr=learning_rate
    )

    print("\n" + "=" * 90)
    print(f"Trial {trial.number} started")
    print("Parameters:")
    print(f"  num_layers: {num_layers}  # fixed")
    print(f"  hidden_size: {hidden_size}")
    print(f"  dropout: {dropout}")
    print(f"  dense_size: {dense_size}")
    print(f"  learning_rate: {learning_rate}")
    print(f"  batch_size: {batch_size}")
    print(f"  patience: {patience}")
    print("=" * 90)

    best_val_loss, best_epoch, best_state_dict = train_model_for_trial(
        model=model_trial,
        train_loader=train_loader_trial,
        val_loader=val_loader_trial,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        max_epochs=80,
        patience=patience
    )

    if best_state_dict is None:
        raise RuntimeError("best_state_dict is None. Check train_model_for_trial().")

    model_trial.load_state_dict(best_state_dict)
    model_trial.eval()

    y_val_scaled_true, y_val_scaled_pred = predict_model(
        model=model_trial,
        data_loader=val_loader_trial,
        device=device
    )

    y_val_true = scaler_y.inverse_transform(y_val_scaled_true)
    y_val_pred = scaler_y.inverse_transform(y_val_scaled_pred)

    val_nse = nse_score(y_val_true, y_val_pred)
    val_kge = kge_score(y_val_true, y_val_pred)
    val_rmse = rmse_score(y_val_true, y_val_pred)
    val_mae = mae_score(y_val_true, y_val_pred)

    elapsed_time = time.time() - start_time

    print("-" * 90)
    print(f"Trial {trial.number} finished")
    print(f"Best epoch    : {best_epoch}")
    print(f"Best val_loss : {best_val_loss:.6f}")
    print(f"Val NSE       : {val_nse:.4f}")
    print(f"Val KGE       : {val_kge:.4f}")
    print(f"Val RMSE      : {val_rmse:.4f}")
    print(f"Val MAE       : {val_mae:.4f}")
    print(f"Elapsed time  : {elapsed_time / 60:.2f} min")
    print("-" * 90)

    try:
        current_best = trial.study.best_value
    except ValueError:
        current_best = None

    if current_best is None or val_nse > current_best:
        torch.save({
            "model_state_dict": best_state_dict,
            "trial_number": trial.number,
            "val_nse": val_nse,
            "val_kge": val_kge,
            "val_rmse": val_rmse,
            "val_mae": val_mae,
            "val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "params": {
                "hidden_size": hidden_size,
                "num_layers": num_layers,
                "dropout": dropout,
                "dense_size": dense_size,
                "learning_rate": learning_rate,
                "batch_size": batch_size,
                "patience": patience
            },
            "feature_cols": feature_cols,
            "target_col": TARGET_COL,
            "window_size": WINDOW_SIZE,
            "forecast_horizon": FORECAST_HORIZON,
            "train_period": [str(time_train.min()), str(time_train.max())],
            "val_period": [str(time_val.min()), str(time_val.max())],
            "test_period": [str(time_test.min()), str(time_test.max())]
        }, BEST_TRIAL_MODEL_PATH)

        print("  -> Best trial model weight saved.")

    return val_nse

In [ ]:
# ============================================================
# 18. Run Optuna
# ============================================================

study = optuna.create_study(
    direction="maximize",
    study_name="Daechung_LSTM_PyTorch_NSE_2layers_pastlevel"
)

study.optimize(
    objective,
    n_trials=5
)

print("\nBest Validation NSE:", study.best_value)
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"{key}: {value}")

print("\nBest trial model checkpoint saved at:")
print(BEST_TRIAL_MODEL_PATH)

In [ ]:
study.optimize(
    objective,
    n_trials=25
)

In [ ]:
# ============================================================
# 19. Load Saved Best Trial Model
# ============================================================

best_checkpoint = torch.load(
    BEST_TRIAL_MODEL_PATH,
    map_location=device,
    weights_only=False
)

print("Best trial:", best_checkpoint["trial_number"])
print("Best epoch:", best_checkpoint["best_epoch"])
print("Best val_loss:", best_checkpoint["val_loss"])
print("Best val NSE:", best_checkpoint["val_nse"])
print("Best val KGE:", best_checkpoint["val_kge"])
print("Best val RMSE:", best_checkpoint["val_rmse"])
print("Best val MAE:", best_checkpoint["val_mae"])
print("Best params:", best_checkpoint["params"])

best_params = best_checkpoint["params"]

best_model = LSTMRegressor(
    input_size=INPUT_SIZE,
    hidden_size=best_params["hidden_size"],
    num_layers=best_params["num_layers"],
    dropout=best_params["dropout"],
    dense_size=best_params["dense_size"]
).to(device)

best_model.load_state_dict(best_checkpoint["model_state_dict"])
best_model.eval()

print("Best trial model loaded.")

In [ ]:
# ============================================================
# 20. Validation & Test Evaluation
# ============================================================

val_loader_best = DataLoader(
    val_dataset,
    batch_size=best_params["batch_size"],
    shuffle=False,
    num_workers=0
)

test_loader_best = DataLoader(
    test_dataset,
    batch_size=best_params["batch_size"],
    shuffle=False,
    num_workers=0
)

y_val_scaled_true, y_val_scaled_pred = predict_model(
    model=best_model,
    data_loader=val_loader_best,
    device=device
)

y_val_true = scaler_y.inverse_transform(y_val_scaled_true)
y_val_pred = scaler_y.inverse_transform(y_val_scaled_pred)

val_metrics = evaluate_predictions(
    y_true=y_val_true,
    y_pred=y_val_pred,
    name="Validation Best Trial LSTM"
)

y_test_scaled_true, y_test_scaled_pred = predict_model(
    model=best_model,
    data_loader=test_loader_best,
    device=device
)

y_test_true = scaler_y.inverse_transform(y_test_scaled_true)
y_test_pred = scaler_y.inverse_transform(y_test_scaled_pred)

test_metrics = evaluate_predictions(
    y_true=y_test_true,
    y_pred=y_test_pred,
    name="Test Best Trial LSTM"
)

In [ ]:
# ============================================================
# 21. Save Scalers and Metrics
# ============================================================

SCALER_X_PATH = MODEL_DIR / "daechung_scaler_X_window72_pastlevel.pkl"
SCALER_Y_PATH = MODEL_DIR / "daechung_scaler_y_window72_pastlevel.pkl"

joblib.dump(scaler_X, SCALER_X_PATH)
joblib.dump(scaler_y, SCALER_Y_PATH)

metrics_summary = {
    "validation": val_metrics,
    "test": test_metrics,
    "best_params": best_params,
    "feature_cols": feature_cols,
    "target_col": TARGET_COL,
    "window_size": WINDOW_SIZE,
    "forecast_horizon": FORECAST_HORIZON
}

METRICS_PATH = METRIC_DIR / "daechung_lstm_window72_2layers_pastlevel_metrics.pkl"
joblib.dump(metrics_summary, METRICS_PATH)

print("Scalers saved:")
print(SCALER_X_PATH)
print(SCALER_Y_PATH)
print("Metrics saved:")
print(METRICS_PATH)

In [ ]:
# ============================================================
# 22. Plot Prediction
# ============================================================

plt.figure(figsize=(14, 5))
plt.plot(time_test, y_test_true, label="Observed")
plt.plot(time_test, y_test_pred, label="Predicted")
plt.xlabel("Time")
plt.ylabel("Daechung Dam Water Level")
plt.title("Observed vs Predicted: Test Set")
plt.legend()
plt.grid(True)
plt.tight_layout()

FIG_PATH = FIGURE_DIR / "daechung_lstm_test_prediction_window72_pastlevel.png"
plt.savefig(FIG_PATH, dpi=300)
plt.show()

print("Figure saved:", FIG_PATH)